In [ ]:
import os
import pickle
import csv
import re
import random
from collections import defaultdict
import torch
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import json

In [ ]:
decoding_alg = "greedy" 

#Parametri configurabili
contextToDo= ["context-200"]

pii_types = ['twitter', 'email_cc', 'phone'] 

languages = ["eng", "fr", "it", "sp", 'de']


results_folders = [
    ('./data', f"./results", 'original'),
    ('./data_p', f"./results_p", 'para')
]



BATCH_SIZE = 32


redo = False

pd.set_option('display.max_colwidth', None)


In [ ]:
def load_json(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            return json.load(f)

def save_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

In [ ]:
def load_data(pii_type, filepath):
    """Carica e preprocessa un dataset PII."""
    data = Dataset.load_from_disk(filepath)
    data = pd.DataFrame(data)
    data['context'] = data['context'].apply(str.strip)
    # Sampling per dataset URL se troppo grande
    if len(data) > 4550 and pii_type == 'url':
        data = data.sample(n=4550, random_state=42).reset_index(drop=True)

    data = Dataset.from_pandas(data[['pii', 'context']])
    return data


def load_pickle(filename):
    """Carica un file pickle."""
    with open(filename, "rb") as pickle_handler:
        results = pickle.load(pickle_handler)
    return results

In [ ]:
def load(pii_type, dataset_path):
    data = load_data(pii_type, dataset_path)
    return data

In [ ]:
for data_folder, _, _ in results_folders:
    print(data_folder)
    for pii_type in pii_types:
        dataset_path=f"{data_folder}/Dataset-{pii_type}-eng" if pii_type != 'twitter' else f"{data_folder}/Dataset-{pii_type}-eng"
        data = load(pii_type, dataset_path)
        print(pii_type, len(data))

In [ ]:
redo

In [ ]:
from time import sleep

In [ ]:
import tqdm

In [ ]:
model_types = {
    'Qwen2.5': ["7B", '3B'], 
     'Llama-3.2': ['1B', '3B'],
     'gpt-j': ['6B'],
     'gpt-neo': ['1.3B', '2.7B']
}

results = {}



    
for pii_type in tqdm.tqdm(pii_types):
    results[pii_type] = {}
    for x in contextToDo:
        results[pii_type][x] = {}
        for model_type in model_types:
            for model_size in model_types[model_type]:

                for (data_folder, result_folder, exp) in results_folders:
                    results[pii_type][x][f"{model_type}-{model_size}-{exp}"] = []
                    
                    for lang in languages:
                        row = {}
                        dataset_path=f"{data_folder}/Dataset-{pii_type}-{lang}"
                        if not os.path.exists(dataset_path):
                            print("Data not available at", dataset_path)
                            row['lang'] = lang
                            row['generated'] = None
                            row['leak'] = None
                            results[pii_type][x][f"{model_type}-{model_size}"].append(row)
                            continue 
                        data = load(pii_type, dataset_path)
                    
                    
                        folder_name=f'{result_folder}/{pii_type}-{lang}'
                
                        context_results_dir = f'{folder_name}/results-{x}'
                        generated_file = f"{context_results_dir}/generated-{model_type}-{model_size}-{decoding_alg}.json"
                        correct_file = f"{context_results_dir}/correct-{model_type}-{model_size}-{decoding_alg}.json"
                        if not os.path.exists(generated_file):
                            print("Results not read at", generated_file)
                            row['lang'] = lang
                            row['generated'] = None
                            row['leak'] = None
                            results[pii_type][x][f"{model_type}-{model_size}-{exp}"].append(row)
                            continue
                            
                        generated = load_json(generated_file) 
                        pii_found = load_json(correct_file)
    
                        row['lang'] = lang
                        row['generated'] = len(generated)
                        row['leak'] = len(pii_found)
        
                        results[pii_type][x][f"{model_type}-{model_size}-{exp}"].append(row)

In [ ]:
lang_order = languages
metric_order = ["leak", "generated"]

In [ ]:
dfs = {}
for pii_type in results:
    context = 'context-200'
    #for context in results[pii_type]:
    dfs[pii_type] = {}
    for model_name in results[pii_type][context]:
         dfs[pii_type][model_name] = pd.DataFrame(results[pii_type][context][model_name])
         #display(dfs[pii_type][model_name])
    dfs[pii_type] = pd.concat(dfs[pii_type]).reset_index()
    dfs[pii_type] = dfs[pii_type].rename(columns={'level_0': 'model'})
    dfs[pii_type] = dfs[pii_type][['model', 'lang', 'generated', 'leak']]
    
    dfs[pii_type] = (
        dfs[pii_type].pivot(
            index="model",
            columns="lang",
            values=["leak", "generated"]
        )
        .swaplevel(0, 1, axis=1)
        .reindex(
            columns=pd.MultiIndex.from_product(
                [lang_order, metric_order],
                names=["lang", None]
            )
        )
    )
    print("*"*80)
    print(pii_type)
    print("*"*80)
    display(dfs[pii_type])

In [ ]:
to_print = pd.concat(dfs)
to_print = to_print.reset_index()
to_print = to_print.rename(columns={'level_0':'PII type'})
to_print = to_print[to_print['model'].str.contains('-original')]
to_print['model'] = to_print['model'].apply(lambda x: x.replace('-original', '').replace('-', ' '))
to_print = to_print.set_index(['PII type', 'model'])
to_print

In [ ]:
pd.concat(dfs).columns

In [ ]:
to_plot_para = pd.concat(dfs)
to_plot_para = to_plot_para[[(l,c) for (l,c) in to_plot_para.columns if c == 'leak']]
to_plot_para.columns = [l for (l,c) in to_plot_para.columns if c == 'leak']
to_plot_para = to_plot_para.reset_index()
to_plot_para = to_plot_para.rename(columns={'level_0':'PII type'})
to_plot_para['conf'] = to_plot_para['model'].apply(lambda x: x.split('-')[-1])
to_plot_para['model'] = to_plot_para['model'].apply(lambda x: x.replace('-original', '').replace('-para', '').replace('-', ' '))
#to_plot_para = to_plot_para.set_index(['PII type', 'model', ])
to_plot_para = to_plot_para.fillna(-1)
to_plot_para

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# ---------------------------------------------------------
# Okabe-Ito colour-blind friendly palette
# ---------------------------------------------------------

LANG_COLORS = {
    "fr": "#0072B2",   # blue
    "it": "#E69F00",   # orange
    "sp": "#009E73",   # green
    "de": "#CC79A7",   # purple
    "eng": "#D55E00",  # vermillion
}

LANG_LABELS = {
    "eng": "English",
    "fr": "French",
    "it": "Italian",
    "sp": "Spanish",
    "de": "German",
}

languages = ["eng", "fr", "it", "sp", "de"]

models = [
    "Llama 3.2 1B",
    "Llama 3.2 3B",
    "Qwen2.5 3B",
    "Qwen2.5 7B",
    "gpt j 6B",
    "gpt neo 1.3B",
    "gpt neo 2.7B",
]

pii_types = ["twitter", "email_cc", "phone"]

MODEL_LABELS = {
    "Llama 3.2 1B": "Llama 3.2\n1B",
    "Llama 3.2 3B": "Llama 3.2\n3B",
    "Qwen2.5 3B": "Qwen2.5\n3B",
    "Qwen2.5 7B": "Qwen2.5\n7B",
    "gpt j 6B": "GPT-J\n6B",
    "gpt neo 1.3B": "GPT-Neo\n1.3B",
    "gpt neo 2.7B": "GPT-Neo\n2.7B",
}


# ---------------------------------------------------------
# Make sure values are numeric
# ---------------------------------------------------------

for col in languages:
    if isinstance(to_plot_para[col], pd.DataFrame):
        # In case duplicate column names exist
        to_plot_para[col] = to_plot_para[col].iloc[:, 0]

    to_plot_para[col] = pd.to_numeric(
        to_plot_para[col],
        errors="coerce"
    )


# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6),
    sharey=False,
)

x = np.arange(len(models))

group_width = 0.90
n_bars = len(languages) * 2
bar_width = group_width / n_bars


for ax, pii_type in zip(axes, pii_types):

    data = to_plot_para[
        to_plot_para["PII type"] == pii_type
    ]

    for lang_idx, lang in enumerate(languages):

        color = LANG_COLORS[lang]

        for cond_idx, condition in enumerate(
            ["original", "para"]
        ):

            values = []

            for model in models:

                rows = data[
                    (data["model"] == model)
                    & (data["conf"] == condition)
                ]

                if rows.empty:
                    value = np.nan
                else:
                    value = rows.iloc[0][lang]

                values.append(value)

            values = np.asarray(values, dtype=float)

            offset = (
                -group_width / 2
                + (lang_idx * 2 + cond_idx) * bar_width
                + bar_width / 2
            )

            ax.bar(
                x + offset,
                values,
                width=bar_width * 0.90,
                color=color,
                edgecolor="black",
                linewidth=0.5,
                hatch="" if condition == "original" else "//",
            )

    # -----------------------------------------------------
    # Formatting
    # -----------------------------------------------------

    ax.set_title(
        pii_type.replace("_cc", "").title(),
        fontsize=14,
        fontweight="bold",
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [MODEL_LABELS[m] for m in models],
        fontsize=9,
    )

    ax.set_ylabel("Number of leaked items")

    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.7,
        alpha=0.25,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ---------------------------------------------------------
# Legends
# ---------------------------------------------------------

language_handles = [
    Patch(
        facecolor=LANG_COLORS[lang],
        edgecolor="black",
        linewidth=0.5,
        label=LANG_LABELS[lang],
    )
    for lang in languages
]

condition_handles = [
    Patch(
        facecolor="white",
        edgecolor="black",
        linewidth=0.7,
        hatch="",
        label="Strict translation",
    ),
    Patch(
        facecolor="white",
        edgecolor="black",
        linewidth=0.7,
        hatch="//",
        label="Paraphrase",
    ),
]


# Language legend
fig.legend(
    handles=language_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.04),
    ncol=5,
    frameon=False,
)


# Original / paraphrase legend
fig.legend(
    handles=condition_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.975),
    ncol=2,
    frameon=False,
)


plt.tight_layout(
    rect=[0, 0, 1, 0.88]
)
plt.savefig('')
plt.show()

In [ ]:
novelty_results = {}

for pii_type in pii_types: 
    if not os.path.exists(f'./results-novelty/{pii_type}/novelty_results.csv'):
        print(f'./results-novelty/{pii_type}/novelty_results.csv not available')
        continue
    novelty_results[pii_type] = pd.read_csv(f'./results-novelty/{pii_type}/novelty_results.csv')
    print("*"*80)
    print(pii_type)
    print("*"*80)
    display(novelty_results[pii_type].describe())

In [ ]:
pd.concat({pii_type: novelty_results[pii_type].describe() for pii_type in novelty_results})